# Пилот синтетического эксперимента

Цели проекта — подготовить сбор данных к A/B-тесту, сравнить четыре модели для предварительного выбора и проверить платформу. Здесь проектируем выборку и смотрим по одному примеру каждого сценария.

## Выборка

1 000 пользователей, 90 дней. В первые 60 дней выходят восемь публикаций. Следующие 30 дней продолжаем наблюдать продажи. Два курса — «Аналитика старт» и «Аналитика про». Исторические суммы по позициям этих курсов служат основой чеков. Они могут уже включать скидки или доли пакета, поэтому исторический прайс из них не восстанавливаем.

Одинаковые пользователи, расписание, затраты и заранее сгенерированные случайные числа во всех сценариях. Касания к недоступному продукту в сценарии запуска исключаются. Каждый пост указывает один курс. Категории — обычная публикация, скидка, анонс события и запуск. В пилоте все восемь размещений условно платные, по 6 000 ₽. Тип публикации хранится отдельно от способа размещения.

## Четыре сценария

| Сценарий | Что меняется относительно ровного спроса |
|---|---|
| Ровный спрос | Оба курса доступны. Базовая вероятность покупки каждого курса — 0,1% на пользователя в день. После рекламного касания к этому курсу прибавляем 0,3 п.п. в течение семи дней, включая день касания. |
| Внешнее событие | Дни 31–37: вероятность фоновой покупки «Аналитика про» умножается на 3. Публикации и рекламная прибавка те же. |
| Запуск курса | «Аналитика про» доступен с дня 31. Дни 31–37: фон для этого курса умножается на 3. Пост дня 31 получает категорию «запуск». Дата запуска условная. |
| Нулевой эффект | Ровный спрос и то же расписание касаний. Рекламная прибавка и снижение цены отключены. |

## Предположения генератора

- Вероятность касания каждого доступного размещения 16%. После рекламного касания вероятность просмотра 65%, после просмотра вероятность заявки 25%. У 30% пользователей в среднем есть один заход без метки.
- Заявки и оплаты генерируются отдельно. Таблица ниже считает заявки пользователей, которые совсем не купили за период. Для сверки оплат по конкретной заявке понадобится отдельная связка и время событий.
- Не больше двух заказов у пользователя, интервал более трёх дней. Фоновая вероятность повторной покупки уменьшается в четыре раза. Два курса в одном заказе возможны с вероятностью 10% при доступности обоих. Сумма заказа равна сумме позиций.
- Снижение цены на 15% действует для покупателя с касанием к скидочному посту о соответствующем курсе за последние семь дней. Скидка применяется к выбранной исторической сумме как условный сценарий. Рекламная прибавка к вероятности покупки одинакова для всех категорий.
- Шаг генерации — день. Внутри дня публикация и касание считаются доступными к моменту покупки. Эти таблицы предназначены для проверки выборки. Для отчёта бота понадобятся точные времена и связи со схемой БД.
- Семидневный срок воздействия в генераторе и 30-дневное окно атрибуции — разные параметры. Скрытая метка заказа описывает ветку генерации. В многокурсовом заказе она относится к покупке, инициировавшей заказ. Все вероятности, затраты и величины всплесков заданы для пилота.



In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/base.csv').exists())
sys.path.insert(0, str(ROOT / 'src'))
from experiments.scenario_pilot import generate_pilots, summarize

pilots = dict(generate_pilots(ROOT / 'data/raw/base.csv', seed=42, n_users=1000))
summary = summarize(pilots.items())
summary


,scenario,users,visitors,touches,views,leads,leads_from_nonbuyers,buyers,orders,items,revenue_rub,ad_tagged_orders
0,flat,1000,835,1623,828,205,167,182,185,209,1542859.75,23
1,external_event,1000,835,1623,828,205,161,197,202,225,1672171.25,26
2,course_launch,1000,807,1458,715,176,142,162,166,184,1362644.75,17
3,zero_ad_effect,1000,835,1623,828,205,172,163,166,186,1384796.50,0


Один прогон помогает проверить объём, долю покупателей, наличие заявок без покупок и состав заказов. Для проверки календаря смотрим продажи по курсам и дням. Различия между четырьмя пилотами описывают условия симуляции. Следующий этап — проверка расчётов и отчёта бота, затем сравнение моделей на повторных прогонах.


In [2]:
import pandas as pd

daily_course_items = pd.concat([
    tables['items'].merge(tables['orders'][['order_id', 'day']], on='order_id', validate='many_to_one')
    .groupby(['day', 'course']).size().rename('items').reset_index().assign(scenario=name)
    for name, tables in pilots.items()
], ignore_index=True)
daily_course_items


,day,course,items,scenario
0,1,Аналитика старт,2,flat
1,2,Аналитика старт,2,flat
2,3,Аналитика про,1,flat
3,4,Аналитика старт,3,flat
4,5,Аналитика про,1,flat
...,...,...,...,...
455,88,Аналитика старт,1,zero_ad_effect
456,89,Аналитика про,1,zero_ad_effect
457,89,Аналитика старт,1,zero_ad_effect
458,90,Аналитика про,1,zero_ad_effect
